# 19. Distributed Training with Keras

### Overview

The `tf.distribute.Strategy` API provides an abstraction for distributing your training across multiple processing units. It allows you to carry out distributed training using existing models and training code with minimal changes.

This tutorial demonstrates how to use the `tf.distribute.MirroredStrategy` to perform in-graph replication with _synchronous training on many GPUs on one machine_. The strategy essentially copies all of the model's variables to each processor. Then, it uses all-reduce to combine the gradients from all processors, and applies the combined value to all copies of the model.

You will use the `tf.keras` APIs to build the model and `Model.fit` for training it.

`MirroredStrategy` trains your model on multiple GPUs on a single machine. For _synchronous training on many GPUs on multiple workers_, use the `tf.distribute.MultiWorkerMirroredStrategy` with the Keras Model.fit or a custom training loop. For other options, refer to the Distributed training guide.

To learn about various other strategies, there is the Distributed training with TensorFlow guide.

### Setup

In [1]:
import tensorflow_datasets as tfds
import tensorflow as tf

import os

# Load the TensorBoard notebook extension
%load_ext tensorboard

In [2]:
print(tf.__version__)

2.20.0


### Download the dataset

Load the MNIST dataset from TensorFlow Datasets. This returns a dataset in the `tf.data` format.

Setting with `with_info` argument to `True` includes the metadata for the entire dataset, which is being saved here to `info`. Among other things, this metadata object includes the number of traing and test examples.

In [3]:
datasets, info = tfds.load(name='mnist', with_info=True, as_supervised=True)

mnist_train, mnist_test = datasets['train'], datasets['test']

C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Dl Completed...: 0 url [00:00, ? url/s]
Dl Size...: 0 MiB [00:00, ? MiB/s]

Dl Completed...:   0%|          | 0/1 [00:00<?, ? url/s]
Dl Size...: 0 MiB [00:00, ? MiB/s]

Dl Completed...:   0%|          | 0/2 [00:00<?, ? url/s]
Dl Size...: 0 MiB [00:00, ? MiB/s]

Dl Completed...:   0%|          | 0/3 [00:00<?, ? url/s]
Dl Size...: 0 MiB [00:00, ? MiB/s]

Dl Completed...:   0%|          | 0/4 [00:00<?, ? url/s]
Dl Size...: 0 MiB [00:00, ? MiB/s]

Dl Size...:   0%|          | 0/9 [00:02<?, ? MiB/s]

Dl Size...:   0%|          | 0/9 [00:02<?, ? MiB/s]

Dl Size...:   0%|          | 0/10 [00:02<?, ? MiB/s]

Dl Size...:   0%|          | 0/10 [00:02<?, ? MiB/s]

Dl Size...:   0%|          | 0/10 [00:02<?, ? MiB/s]

Extrac

Dataset mnist downloaded and prepared to C:\Users\yigit\tensorflow_datasets\mnist\3.0.1. Subsequent calls will reuse this data.


### Define the distribution strategy

Create a `MirroredStrategy` object. This will handle distribution and provide a context manager (`MirroredStrategy.scope`) to build your model inside.

In [4]:
strategy = tf.distribute.MirroredStrategy()

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)


In [5]:
print("Number of devices: {}".format(strategy.num_replicas_in_sync))

Number of devices: 1


### Setp up the input pipeline

When training a model with multiple GPUs, you can use the extra computing power effectively by increasing the batch size. In general, use the largest batch size that fits the GPU memory and tune the learning rate accordingly.

In [6]:
# You can also do info.splits.total_num_examples to get the total number of examples in the dataset.
num_train_examples = info.splits['train'].num_examples
num_test_examples = info.splits['test'].num_examples

BUFFER_SIZE = 10000

BATCH_SIZE_PER_REPLICA = 64
BATCH_SIZE = BATCH_SIZE_PER_REPLICA * strategy.num_replicas_in_sync

Define a function that normalizes the image pixel values from the `[0, 255]` range to the `[0, 1]` range (feature scaling):

In [7]:
def scale(image, label):
    image = tf.cast(image, tf.float32)
    image /= 255

    return image, label

Apply this `scale` function to the training and test data, and then use the `tf.data.Dataset` APIs to shuffle the training data (`Dataset.shuffle`), and batch it (`Dataset.batch`). Notice that you are also keeping an in-memory cache of the training data to improve performance (`Dataset.cache`).

In [8]:
train_dataset = mnist_train.map(scale).cache().shuffle(BUFFER_SIZE).batch(BATCH_SIZE)
eval_dataset = mnist_test.map(scale).batch(BATCH_SIZE)

### Create the model and instantiate the optimizer

Within the context of `Strategy.scope`, create and compile the model using the Keras API:

In [9]:
with strategy.scope():
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(28, 28, 1)),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dense(10)
    ])

    model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                  optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  metrics=['accuracy'])

C:\Users\yigit\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


For this toy example with the MNIST dataset, you will be using the Adam optimizer's default learning rate of 0.001.

For larger datasets, the key benefit of distributed training is to learn more in each training step, because each step processes more training data in parallel, which allows for a larger learning rate (within the limits of the model and dataset).

### Define the callbacks

Define the following Keras Callbacks:

* `tf.keras.callbacks.TensorBoard`: writes a log for TensorBoard, which allows you to visualize the graphs.
* `tf.keras.callbacks.ModelCheckpoint`: saves the model at a certain frequency, such as after every epoch.
* `tf.keras.callbacks.BackupAndRestore`: provides the fault tolerance functionality by backing up the model and current epoch number. Learn more in the _Fault tolerance_ section of the Multi-worker training with Keras tutorial.
* `tf.keras.callbacks.LearningRateScheduler`: schedules the learning rate to change after, for example, every epoch/batch.

For illustrative purposes, add a custom callback called `PrintLR` to display the learning rate in the notebook.

**Note:** Use the `BackupAndRestore` callback instead of `ModelCheckpoint` as the main mechanism to restore the training state upon a restart from a job failure. Since `BackupAndRestore` only supports eager mode, in graph mode consider using `ModelCheckpoint`.

In [14]:
# Define the checkpoint directory to store the checkpoints.
checkpoint_dir = './training_checkpoints'
# Define the name of the checkpoint files.
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}.weights.h5")

In [15]:
# Define a function for decaying the learning rate. You can define any decay function you need.
def decay(epoch):
    if epoch < 3:
        return 1e-3
    elif epoch >= 3 and epoch < 7:
        return 1e-4
    else:
        return 1e-5

In [16]:
# Define a callback for printing the learning rate at the end of each epoch.
class PrintLR(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        print("\nLearning rate for epoch {} is {}".format(epoch + 1, model.optimizer.lr.numpy()))

In [19]:
# Put all the callbacks together.
callbacks = [
    tf.keras.callbacks.TensorBoard(log_dir='./logs'),
    tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_prefix, save_weights_only=True),
    tf.keras.callbacks.LearningRateScheduler(decay),
    # PrintLR()
]

### Train and evaluate

Now, train the model in the usual way by calling Keras `Model.fit` on the model and passing in the dataset created at the beginning of the tutorial. This step is the same whether you are distributing the training or not.

In [20]:
EPOCHS = 12

model.fit(train_dataset, epochs=EPOCHS, callbacks=callbacks)

Epoch 1/12
938/938 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.9791 - loss: 0.0703 - learning_rate: 0.0010
Epoch 2/12
938/938 ━━━━━━━━━━━━━━━━━━━━ 26s 25ms/step - accuracy: 0.9854 - loss: 0.0489 - learning_rate: 0.0010
Epoch 3/12
938/938 ━━━━━━━━━━━━━━━━━━━━ 40s 23ms/step - accuracy: 0.9887 - loss: 0.0368 - learning_rate: 0.0010
Epoch 4/12
938/938 ━━━━━━━━━━━━━━━━━━━━ 39s 21ms/step - accuracy: 0.9949 - loss: 0.0198 - learning_rate: 1.0000e-04
Epoch 5/12
938/938 ━━━━━━━━━━━━━━━━━━━━ 18s 19ms/step - accuracy: 0.9961 - loss: 0.0166 - learning_rate: 1.0000e-04
Epoch 6/12
938/938 ━━━━━━━━━━━━━━━━━━━━ 24s 23ms/step - accuracy: 0.9964 - loss: 0.0151 - learning_rate: 1.0000e-04
Epoch 7/12
938/938 ━━━━━━━━━━━━━━━━━━━━ 21s 22ms/step - accuracy: 0.9968 - loss: 0.0139 - learning_rate: 1.0000e-04
Epoch 8/12
938/938 ━━━━━━━━━━━━━━━━━━━━ 23s 24ms/step - accuracy: 0.9976 - loss: 0.0118 - learning_rate: 1.0000e-05
Epoch 9/12
938/938 ━━━━━━━━━━━━━━━━━━━━ 38s 21ms/step - accuracy: 0.9976 - loss: 0.0

Check for saved checkpoints:

In [21]:
# Check the checkpoint directory.
# ls {checkpoint_dir}

To check how well the model performs, load the latest checkpoint and call `Model.evaluate` on the test data:

In [34]:
checkpoint_dir

'./training_checkpoints'

In [43]:
# model.load_weights(tf.train.latest_checkpoint(checkpoint_dir))
print(os.path.exists(checkpoint_dir))
print(os.listdir(checkpoint_dir))

eval_loss, eval_acc = model.evaluate(eval_dataset)

print("Eval loss: {}, Eval accuracy: {}".format(eval_loss, eval_acc))

True
['ckpt_1.weights.h5', 'ckpt_10.weights.h5', 'ckpt_11.weights.h5', 'ckpt_12.weights.h5', 'ckpt_2.weights.h5', 'ckpt_3.weights.h5', 'ckpt_4.weights.h5', 'ckpt_5.weights.h5', 'ckpt_6.weights.h5', 'ckpt_7.weights.h5', 'ckpt_8.weights.h5', 'ckpt_9.weights.h5']
157/157 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9876 - loss: 0.0407
Eval loss: 0.0407094731926918, Eval accuracy: 0.9876000285148621


To visualize the output, launch TensorBoard and view the logs:

In [44]:
%tensorboard --logdir=logs

In [45]:
# ls -sh .logs

### Save the model

Save the model to a `.keras` zip archive using `Model.save`. After your model is saved, you can load it with or without the `Strategy.scope`.

In [46]:
path = 'my_model_dist.keras'

In [47]:
model.save(path)

Now, load the model without `Strategy.scope`:

In [48]:
unreplicated_model = tf.keras.models.load_model(path)

unreplicated_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy'])

eval_loss, eval_acc = unreplicated_model.evaluate(eval_dataset)

print("Eval loss: {},  Eval accuracy: {}".format(eval_loss, eval_acc))

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9876 - loss: 0.0407
Eval loss: 0.0407094731926918,  Eval accuracy: 0.9876000285148621


Load the model with `Strategy.scope`:

In [49]:
with strategy.scope():
    replicated_model = tf.keras.models.load_model(path)
    replicated_model.compile(loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                             optimizer=tf.keras.optimizers.Adam(),
                             metrics=['accuracy'])

    eval_loss, eval_acc = replicated_model.evaluate(eval_dataset)
    print('Eval loss: {}, Eval accuracy: {}'.format(eval_loss, eval_acc))

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9876 - loss: 0.0407
Eval loss: 0.0407094731926918, Eval accuracy: 0.9876000285148621


### Additional resources

More examples that use different distribution with the Keras `Model.fit` API:

1. The Solve GLUE tasks using BERT on TPU tutorial uses `tf.distribute.MirroredStrategy` for training on GPUs and `tf.distribute.TPUStrategy` on TPUs.
2. The Save and load a model using a distribution strategy tutorial demonstrates how to use the SavedModel APIs with `tf.distribute.Strategy`.
3. The official TensorFlow models can be configured to run multiple distribution strategies.

To learn more about TensorFlow distribution strategies:

1. The Custom training with tf.distribute.Strategy tutorial shows how to use the `tf.distribute.MirroredStrategy` for single-worker training with a custom training loop.
2. The Multi-worker training with Keras tutorial shows how to use the `MultiWorkerMirroredStrategy` with `Model.fit`.
3. The Custom training loop with Keras and MultiWorkerMirroredStrategy tutorial shows how to use the `MultiWorkerMirroredStrategy` with Keras and a custom training loop.
4. The Distributed training in TensorFlow guide provides an overview of the available distribution strategies.
5. The Better performance with tf.function guide provides information about other strategies and tools, such as the TensorFlow Profiler you can use to optimize the performance of your TensorFlow models.

**Note:** `tf.distribute.Strategy` is actively under development and TensorFlow will be adding more examples and tutorials in the near future. Please give it a try. Your feedback is welcome - feel free to submit via issues on GitHub.